# walk

> The runtime of one walk step: a short tutor note, then a short-answer check.

A walk step shows a passage, then a short note from the tutor, then one check. The journey (`journey.py`) plans the steps
and picks the check of each step. This module runs a step.

The note tells the learner what to look for in the passage. It does not answer the check. The short-answer check is a normal
tutor question, with two changes: the step decides its way and depth, and each attempt keeps the walk, the step, the section and
the page. A sitting log counts every check, also the checks that do not go through `Session.answer`.

In [ ]:
#| default_exp walk

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re, time
from fastcore.utils import *
from abhyasa.llm import for_task
from abhyasa.prompts import register, prompt
from abhyasa.tutor import Tutor, Session
import abhyasa.journey  # adds Learning.walk_step

## The note

The prompt for the note. It gets the idea, the passage and the kind of the next check. It never gets the check question.

In [ ]:
#| export
NOTE_SP = register('walk.note', 'Walk note', 'How the tutor writes the short note under the passage of each step in a walk.',
    "You write a short note for a learner who just read a passage. Write 2 or 3 sentences, at most 60 words. "
    "Say what the passage says about the idea, in plain words. Do not give the answer to a question on this idea. "
    "End with what to look for next. Reply with the note only.")

The reply is cut to 3 sentences and 60 words, even when the model writes more.

In [ ]:
#| export
def _trim(text:str, n:int=3, words:int=60) -> str:
    "The first `n` sentences of `text`, fewer when they have more than `words` words, and at most `words` words."
    ss = re.split(r'(?<=[.!?])\s+', ' '.join(text.split()))[:n]
    while len(ss) > 1 and len(' '.join(ss).split()) > words: ss.pop()
    return ' '.join(' '.join(ss).split()[:words])

`tutor_note` writes the note once and keeps it on the step. A later call gives the kept note.

In [ ]:
#| export
def tutor_note(lr, ask, j, i:int) -> str:
    "The note for step `i` of walk `j`: kept on the step, else written from the idea and the passage."
    s = (lr.journey(j.id) or j).steps[i]
    if s.get('note'): return s['note']
    c = lr.concept(s['slug'])
    p = (f"Idea: {c.name}\n{c.get('statement', '')}\n\nThe passage:\n{(s.get('read') or {}).get('text') or 'none'}\n\n"
         f"The next check is a {(s.get('check') or {}).get('kind', 'short')} check.")
    n = _trim(for_task(ask, 'hint')(p, prompt('walk.note')))
    lr.walk_step(j.id, i, note=n)
    return n

## A short answer

Each attempt of a step keeps its place in the source. `step_meta` makes that record.

In [ ]:
#| export
def step_meta(j, i:int) -> dict:
    "The walk, the step, the section and the page of step `i`, for the record of each attempt."
    r = j.steps[i].get('read') or {}
    return dict(walk=j.id, step=i, section=r.get('section'), page=r.get('page'))

`open_short` opens the question of a step. The check of the step gives the way and the depth. The note goes into the question prompt. The question id stays on the step.

In [ ]:
#| export
def open_short(tutor:Tutor, j, i:int, session:str) -> AttrDict:
    "Open the short-answer check of step `i` of walk `j`, and keep its id on the step."
    s, lr = j.steps[i], tutor.learning
    ck = s['check']
    q = tutor.ask_on(lr.concept(s['slug']), ck['depth'], ck.get('why', ''), session=session, method=ck['way'],
                     meta=step_meta(j, i), read=tutor_note(lr, tutor.ask, j, i))
    lr.walk_step(j.id, i, qid=q.id)
    return q

## The sitting log

A demo or an order prediction does not go through `Session.answer`. `noted` adds one row for it, in the same shape as the rows of a question.

In [ ]:
#| export
@patch
def noted(self:Session, a:dict, floor0:int):
    "Add one log row for the attempt `a`, made outside `Session.answer` (a demo or an order prediction), and save the sitting."
    c = self.tutor.learning.concept(a['concept'])
    self.log.append(dict(qid='', concept=a['concept'], name=c.name if c else a['concept'], depth=a.get('depth', 2), why=a.get('why', ''),
                         question=a.get('question', ''), method=a.get('method', ''), correct=a.get('correct', False), outcome=a.get('outcome', ''),
                         floor0=floor0, floor=c.get('floor', 0) if c else 0, misconception=a.get('misconception', ''), turns=[], at=time.time()))
    self.save()

## Tests for the step runtime

These tests use the small paper in `tests/walk_data.py` and a fake model.

In [ ]:
#| hide
import sys; sys.path.insert(0, '../tests'); from walk_data import *
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
from abhyasa.tutor import Tutor, Session
from abhyasa.journey import journey_from_source

A walk with one fake model. It gives a note, a grade and a question.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY, 'You write a short note': 'Softmax makes weights from scores. Look at what a large score does.',
            'You grade one answer': dict(correct=True, grade=9, first_error='', misconception='', feedback='Yes.')},
           dflt=dict(question='Why do the weights add to 1?', answer='Each is divided by the sum.', rubric=['divide by the sum']))
j = journey_from_source(lr, ask, did)
t = Tutor(lr, ask, context=False)

The note is kept on its step. A second call makes no model call.

In [ ]:
n = tutor_note(lr, ask, j, 1)
assert n.startswith('Softmax makes') and lr.journey(j.id).steps[1]['note'] == n
assert tutor_note(lr, Fake(), lr.journey(j.id), 1) == n

The note keeps at most 3 sentences and 60 words.

In [ ]:
long = tutor_note(lr, Fake({'You write a short note': 'One. Two. Three. Four.'}), lr.journey(j.id), 0)
assert long == 'One. Two. Three.'
wide = tutor_note(lr, Fake({'You write a short note': ' '.join(['word'] * 40) + '. ' + ' '.join(['more'] * 40) + '.'}), lr.journey(j.id), 2)
assert len(wide.split()) <= 60

The step decides the way and the depth. The question comes without its answer, and the prompt has the note.

In [ ]:
s = Session(t, journey=j.id)
lr.walk_step(j.id, 1, check=dict(way='words', kind='short', depth=2, why='plain words fit'))
q = open_short(t, lr.journey(j.id), 1, s.id); s.q = q
assert q.method == 'words' and q.depth == 2
assert 'answer' not in q and 'rubric' not in q and lr.journey(j.id).steps[1]['qid'] == q.id
assert 'The learner just read this' in ask.calls[-1][0] and n in ask.calls[-1][0]

The attempt has its walk, step, section and page. The sitting log has the close.

In [ ]:
r = s.answer('Each weight is divided by the sum of all.')
a = lr.attempts('softmax', limit=1)[0]
assert r.done and (a.walk, a.step, a.section, a.page) == (j.id, 1, '3.2.1 Scaled Dot-Product Attention', 4)
assert s.log[-1]['concept'] == 'softmax'

The review rule: the last way of a walk check is left out of the next question on the idea.

In [ ]:
lr.update('softmax', due=0)
assert all(t.ask_on(lr.concept('softmax'), 3).method != a.method for _ in range(10))

A demo or an order prediction is not made in `Session.answer`. `noted` adds its row to the sitting log and saves it.

In [ ]:
s.noted(dict(concept='softmax', question='Predict', correct=True, outcome='solid', method='demo', kind='demo'), floor0=2)
assert s.log[-1]['method'] == 'demo' and lr.session(s.id).log[-1]['method'] == 'demo'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()